# Bibliotecas

In [ ]:
import os
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (Input, Embedding, LSTM, GRU, Bidirectional, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Flatten)
from tensorflow.keras.utils import to_categorical

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file

path = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-50d", filename="embeddings.safetensors")

data = load_file(path)
vectors = data["embeddings"]

vocab_path = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-50d",filename="vocab.txt")

with open(vocab_path) as f:
    vocab = [w.strip() for w in f]

print(vectors.shape)


In [ ]:
# Baixar recursos do NLTK (se necessário)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab') # Download the missing 'punkt_tab' package

# Pré-Processamento dos Dados

In [ ]:
# Definir stopwords e inicializar lematizador
stop_words_manchetes = ["em", "de", "e", "mt", "no", "na", "da", "do", "o", "a"]

def normalize_text(text):
    # Converter para minúsculas
    text = text.lower()

    # Remover números e caracteres especiais
    text = re.sub(r'[^a-zA-Zà-úÀ-Ú\s]', '', text)

    # Tokenizar
    tokens = word_tokenize(text)

    # Remover stopwords
    normalized_tokens = [word for word in tokens if word not in stop_words_manchetes]
    print(normalized_tokens)

    # Rejuntar tokens normalizados
    return ' '.join(normalized_tokens)

In [ ]:
df = pd.read_csv('Dataset.csv', sep=',')

In [ ]:
df = df[df['Manchete'].notna()]
df = df[df['Manchete'].str.strip() != '']

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
df['Manchete_preprocessada'] = df['Manchete'].apply(normalize_text)

# Criando a lista de vetores de embedding para cada Manchete

In [ ]:
from tensorflow.keras.preprocessing.text import text_to_word_sequence
import numpy as np

MAX_SEQUENCE_LENGTH = 30
EMBEDDING_DIM = 50
vocab2idx = {w: i for i, w in enumerate(vocab)}

def gerar_matrizes_word2vec(textos):
    matrizes = []

    for manchete in textos:
        tokens = text_to_word_sequence(manchete)
        vetores = []

        for token in tokens:
            idx = vocab2idx.get(token)

            if idx is not None:
                vetor = vectors[idx]
                if len(vetor) == EMBEDDING_DIM:
                    vetores.append(vetor)

        vetores = np.array(vetores)

        if len(vetores) < MAX_SEQUENCE_LENGTH:
            padding = np.zeros((MAX_SEQUENCE_LENGTH - len(vetores), EMBEDDING_DIM))
            vetores = np.vstack([vetores, padding])
        else:
            vetores = vetores[:MAX_SEQUENCE_LENGTH]

        matrizes.append(vetores)

    return np.array(matrizes)

In [ ]:
X = gerar_matrizes_word2vec(df['Manchete_preprocessada'])
y = to_categorical(df['Polaridade'], num_classes=3)

# Desenvolvimento de Modelos

## Validação Cruzada para a Escolha dos Melhores Hiperparâmetros

In [ ]:
import numpy as np
import time
import json
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Bidirectional, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Funções para construir modelos
def construir_lstm(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        LSTM(units1, return_sequences=True),
        LSTM(64),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_gru(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        GRU(units1, return_sequences=True),
        GRU(64),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_bilstm(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        Bidirectional(LSTM(units1, return_sequences=True)),
        Bidirectional(LSTM(64)),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_cnn(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        Conv1D(units1, kernel_size=5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Dicionário de modelos
modelos = {
    'LSTM': construir_lstm,
    'GRU': construir_gru,
    'BiLSTM': construir_bilstm,
    'CNN': construir_cnn
}

# Hiperparâmetros
param_grid = {
    'units_layer_one': [64, 128],
    'dropout': [0.3, 0.5],
    'learning_rate': [0.001, 0.003]
}

# Função principal
def treinar_modelos_com_cv(modelos, param_grid, X_train, y_train, X_test, y_test, k, epochs, batch_size):
    resultados = {}

    for nome_modelo, construtor in modelos.items():
        print(f'\n Testando modelo: {nome_modelo}')
        melhor_score = 0
        melhor_comb = None
        f1_scores_por_combinacao = {}
        inicio_total_modelo = time.time()

        for u1 in param_grid['units_layer_one']:
            for dropout in param_grid['dropout']:
                for lr in param_grid['learning_rate']:
                    print(f'Testando: u1={u1}, dropout={dropout}, lr={lr}')
                    scores = []
                    f1_scores = []

                    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
                    y_strat = np.argmax(y_train, axis=1)

                    for train_idx, val_idx in skf.split(X_train, y_strat):
                        X_t, X_val = X_train[train_idx], X_train[val_idx]
                        y_t, y_val = y_train[train_idx], y_train[val_idx]

                        class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(np.argmax(y_t, axis=1)), y=np.argmax(y_t, axis=1))
                        class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

                        model = construtor(X.shape[1:], y.shape[1], u1, dropout, lr)
                        model.fit(X_t, y_t, epochs=epochs, batch_size=batch_size, verbose=0, class_weight=class_weight_dict)

                        preds = np.argmax(model.predict(X_val, verbose=0), axis=1)
                        y_true = np.argmax(y_val, axis=1)

                        acc = np.mean(preds == y_true)
                        f1 = f1_score(y_true, preds, average='macro')
                        scores.append(acc)
                        f1_scores.append(f1)

                    media_f1 = np.mean(f1_scores)
                    chave = f'u{u1}_d{dropout}_lr{lr}'
                    f1_scores_por_combinacao[chave] = f1_scores

                    print(f'→ F1-Score médio (macro): {media_f1:.4f}')

                    if media_f1 > melhor_score:
                        melhor_score = media_f1
                        melhor_comb = (u1, dropout, lr)

        tempo_total_modelo = time.time() - inicio_total_modelo

        print(f'\n Melhor combinação para {nome_modelo}: {melhor_comb} com F1-score médio: {melhor_score:.4f}')

        # Treinamento final
        print(f'Treinando modelo final {nome_modelo}...')
        u1, dropout, lr = melhor_comb
        class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(np.argmax(y_train, axis=1)), y=np.argmax(y_train, axis=1))
        class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

        model_final = construtor(X.shape[1:], y.shape[1], u1, dropout, lr)
        inicio_treinamento_final = time.time()
        history = model_final.fit(X_train, y_train, epochs=50, batch_size=batch_size,
                                  verbose=1, class_weight=class_weight_dict)
        tempo_treinamento_final = time.time() - inicio_treinamento_final

        preds_test = np.argmax(model_final.predict(X_test, verbose=1), axis=1)
        y_true_test = np.argmax(y_test, axis=1)

        report = classification_report(y_true_test, preds_test, output_dict=True)
        cm = confusion_matrix(y_true_test, preds_test).tolist()

        # Acc & F1 macro final do modelo
        acc_final = np.mean(preds_test == y_true_test)
        f1_macro_final = report['macro avg']['f1-score']
        print(f'F1-score (macro) final do modelo {nome_modelo}: {f1_macro_final:.4f}')
        print(f'Acurácia final do modelo {nome_modelo}: {acc_final:.4f}')

        print(f'Resultados do modelo final ({nome_modelo}):\n{classification_report(y_true_test, preds_test)}')

        resultados[nome_modelo] = {
            'melhores_hiperparametros': {'units': u1, 'dropout': dropout, 'lr': lr},
            'tempo_total_combinacoes_e_cv': tempo_total_modelo,
            'tempo_treinamento_final': tempo_treinamento_final,
            'acc_final': acc_final,
            'f1_score_macro_final': f1_macro_final,
            'classification_report': report,
            'matriz_confusao': cm,
            'acc_por_epoca': history.history['accuracy'],
            'loss_por_epoca': history.history['loss'],
            'f1_scores_por_combinacao': f1_scores_por_combinacao
        }


    # Salvar em JSON
    caminho = "resultados_modelos_WV.json"
    with open(caminho, "w") as f:
        json.dump(resultados, f, indent=4)

    # Fazer download no Colab
    try:
        from google.colab import files
        files.download(caminho)
    except:
        print("Não Disponível")


# Executar
treinar_modelos_com_cv(
    modelos=modelos,
    param_grid=param_grid,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    k=10,
    epochs=30,
    batch_size=128
)

## Modelos Finais com os Melhores Hiperparâmetros (Célula usada para salvar os Modelos)

In [ ]:
import numpy as np
import pandas as pd
import time
import json
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, GRU, Bidirectional, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical

# Cria diretórios para salvar resultados
os.makedirs("modelos_treinados", exist_ok=True)


# Arquitetura dos Modelos
def construir_lstm(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        LSTM(units1, return_sequences=True),
        LSTM(64),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_gru(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        GRU(units1, return_sequences=True),
        GRU(64),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_bilstm(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        Bidirectional(LSTM(units1, return_sequences=True)),
        Bidirectional(LSTM(64)),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def construir_cnn(input_shape, num_class, units1, dropout, learning_rate):
    model = Sequential([
        Input(shape=(input_shape)),
        Conv1D(units1, kernel_size=5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(64, activation='relu'),
        Dropout(dropout),
        Dense(num_class, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate), loss='categorical_crossentropy', metrics=['accuracy'])
    return model


# Descomentar apenas o modelo e hiperparâmetros que serão utilizados
modelos = {
    #'LSTM': construir_lstm,
    #'GRU': construir_gru,
    #'BiLSTM': construir_bilstm,
    #'CNN': construir_cnn
}

hiperparametros_finais = {
    #'LSTM': {'units': 128, 'dropout': 0.3, 'lr': 0.003},
    #'GRU': {'units': 128, 'dropout': 0.3, 'lr': 0.003},
    #'BiLSTM': {'units': 128, 'dropout': 0.3, 'lr': 0.003},
    #'CNN': {'units': 128, 'dropout': 0.3, 'lr': 0.003}
}


# Função de Treino
def treinar_modelos_finais(modelos, hiperparametros, X_train, y_train, X_test, y_test,
                          epochs, batch_size):

    resultados = {}

    for nome_modelo, construtor in modelos.items():
        print(f'\n Treinando modelo final: {nome_modelo}')

        # Hiperparâmetros
        u1 = hiperparametros[nome_modelo]['units']
        dropout = hiperparametros[nome_modelo]['dropout']
        lr = hiperparametros[nome_modelo]['lr']

        # Pesos de classe
        class_weights = compute_class_weight(
            class_weight='balanced',
            classes=np.unique(np.argmax(y_train, axis=1)),
            y=np.argmax(y_train, axis=1)
        )
        class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

        # Construção do modelo
        model = construtor(X_train.shape[1:], y_train.shape[1], u1, dropout, lr)

        # Treinamento
        inicio = time.time()
        history = model.fit(
            X_train, y_train,
            epochs=epochs,
            batch_size=batch_size,
            verbose=1,
            class_weight=class_weight_dict,
            validation_data=(X_test, y_test)
        )
        tempo_treinamento = time.time() - inicio

        # Salvar modelo final
        model.save(f"modelos_treinados/modelo_{nome_modelo}.h5")

        # Avaliação SOMENTE MÉTRICAS
        preds_probs = model.predict(X_test, verbose=0)
        preds_test = np.argmax(preds_probs, axis=1)
        y_true_test = np.argmax(y_test, axis=1)

        report = classification_report(y_true_test, preds_test, output_dict=True)
        cm = confusion_matrix(y_true_test, preds_test).tolist()

        acc_final = np.mean(preds_test == y_true_test)
        f1_macro_final = report['macro avg']['f1-score']

        print(f'{nome_modelo} - Acc final: {acc_final:.4f}, F1-macro: {f1_macro_final:.4f}')

        resultados[nome_modelo] = {
            'hiperparametros': {'units': u1, 'dropout': dropout, 'lr': lr},
            'tempo_treinamento': tempo_treinamento,
            'acc_final': acc_final,
            'f1_macro_final': f1_macro_final,
            'classification_report': report,
            'matriz_confusao': cm,
            'acc_por_epoca': history.history['accuracy'],
            'loss_por_epoca': history.history['loss'],
            'val_acc_por_epoca': history.history.get('val_accuracy', []),
            'val_loss_por_epoca': history.history.get('val_loss', [])
        }

    with open("resultados_modelos_finais_Word2Vec.json", "w", encoding='utf-8') as f:
        json.dump(resultados, f, indent=4, ensure_ascii=False)

    return resultados

# Divisão em Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Treinar os modelos
resultados = treinar_modelos_finais(
    modelos=modelos,
    hiperparametros=hiperparametros_finais,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    epochs=50,
    batch_size=128
)

print("\n" + "="*70)
print("Treinamento Concluído!")
print("="*70)
print("Arquivos gerados:")
print("   - modelos_treinados/modelo_*.h5")
print("   - resultados_modelos_finais_Word2Vec.json")
print("="*70)

In [ ]:
!zip -r modelos_treinados.zip modelos_treinados//

In [ ]:
from google.colab import files
files.download('modelos_treinados.zip')

# Predição de Novos Dados

In [ ]:
from tensorflow.keras.models import load_model

# Mudar o caminho a depender do modelo
modelo = load_model("modelos_treinados/modelo_CNN.h5")

df_samples = pd.read_csv("Samples.csv")
df_samples['Manchete_preprocessada'] = df_samples['Manchete'].apply(normalize_text)

embeddings_novos = gerar_matrizes_word2vec(df_samples['Manchete_preprocessada'])

preds = modelo.predict(embeddings_novos)

classes = np.argmax(preds, axis=1)


In [ ]:
classes